# **YOLO V8 Model Training**




# **Step 1 : Dual GPU Verification**

In [1]:
!nvidia-smi

Sun Jan 25 06:37:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Step 2 : Dependencies Installation**

In [2]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


# **Step 3 : Dataset / Directory Inspection**

In [3]:
import os

# Define the path where the dataset was downloaded by kagglehub
dataset_base_path = "/kaggle/input/indian-driving-dataset-detections-yolov11"

print(f"Inspecting directory structure at: {dataset_base_path}")

# List contents of the base directory
print(f"\nContents of {dataset_base_path}:")
for item in os.listdir(dataset_base_path):
    print(item)

# Subdirectory Inspection
dataset_subdir = os.path.join(dataset_base_path, "IDDDetectionsYOLODataset")
if os.path.exists(dataset_subdir):
    print(f"\nContents of {dataset_subdir}:")
    for item in os.listdir(dataset_subdir):
        print(item)
    # Inspecting train, val & test directory by appending 1st 5 items
    expected_subdirs = ['train', 'val', 'test']
    for subdir in expected_subdirs:
        images_path = os.path.join(dataset_subdir, subdir, 'images')
        labels_path = os.path.join(dataset_subdir, subdir, 'labels')
        if os.path.exists(images_path):
            print(f"\nContents of {images_path} (first 5 items):")
            try:
                for i, item in enumerate(os.listdir(images_path)):
                    if i < 5:
                        print(item)
                    else:
                        break
            except Exception as e:
                print(f"Could not list contents: {e}")
        else:
            print(f"\n{images_path} not found.")

        if os.path.exists(labels_path):
            print(f"\nContents of {labels_path} (first 5 items):")
            try:
                for i, item in enumerate(os.listdir(labels_path)):
                    if i < 5:
                        print(item)
                    else:
                        break
            except Exception as e:
                print(f"Could not list contents: {e}")
        else:
            print(f"\n{labels_path} not found.")

Inspecting directory structure at: /kaggle/input/indian-driving-dataset-detections-yolov11

Contents of /kaggle/input/indian-driving-dataset-detections-yolov11:
IDDDetectionsYOLODataset

Contents of /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset:
license.md
ReadMe.md
data.yaml
val
test
train

Contents of /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/train/images (first 5 items):
BLR-2018-05-17_15-56-30_sideLeft_000762_r.jpg
BLR-2018-05-29_11-08-34_sideLeft_000213_r.jpg
BLR-2018-05-29_10-59-01_sideRight_000780_r.jpg
BLR-2018-06-20-07-01-47_part_14_0000431.jpg
BLR-2018-05-17_16-32-30_frontNear_0000600.jpg

Contents of /kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/train/labels (first 5 items):
BLR-2018-05-11_13-42-30_sideRight_right_0000583.txt
BLR-2018-05-22_11-54-07_sideLeft_000483_r.txt
BLR-2018-05-22_12-04-54_sideLeft_0010185.txt
HYD-2018-06-11_13-54-41_frontNear_left_0001785.txt
BL

# **Step 5 : Model Configuration & Training**

### **No. of Epochs : 5 Epoch**

In [ ]:
from ultralytics import YOLO
import os
import shutil
import torch

# Update Data.yaml path
data_yaml_path = '/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset/data.yaml'

# Define the dataset directory path
dataset_base_path = "/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset"

#cache_dir 
cache_dir = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)

# Writable folder for YOLO cache
os.makedirs("/kaggle/working/cache", exist_ok=True)

# Writable folder for PyTorch CUDA kernels
os.makedirs("/kaggle/working/torch_cache", exist_ok=True)
os.environ["TORCH_KERNEL_CACHE_PATH"] = "/kaggle/working/torch_cache"

# Initialize the YOLO model
model = YOLO('yolov8m.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 5, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.005,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.1,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 0.05,
    'cls': 0.5,
    'dfl': 1.0,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 10, # Disable mosaic for the last 10 epochs
    'cache': cache_dir,
    'workers': 3,
    'device': list(range(torch.cuda.device_count()))
              if torch.cuda.device_count() > 1 else 0
}

# Start training
results = model.train(**train_args)

print("Training complete.")

# Model Export 
# Paths
folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/5_epoch_training_output"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")


### **No. of Epochs : 10 Epoch**

In [ ]:
from ultralytics import YOLO
import os
import shutil
import torch

# Update Data.yaml path
data_yaml_path = '/kaggle/input/kaggle-yaml/kaggle_new_data.yaml'

# Define the dataset directory path
dataset_base_path = "/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset"

#cache_dir 
cache_dir = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)

# Writable folder for YOLO cache
os.makedirs("/kaggle/working/cache", exist_ok=True)

# Writable folder for PyTorch CUDA kernels
os.makedirs("/kaggle/working/torch_cache", exist_ok=True)
os.environ["TORCH_KERNEL_CACHE_PATH"] = "/kaggle/working/torch_cache"

# Initialize the YOLO model
model = YOLO('yolov8m.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 10, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.005,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.1,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 0.05,
    'cls': 0.5,
    'dfl': 1.0,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 10, # Disable mosaic for the last 10 epochs
    'cache': cache_dir,
    'workers': 3,
    'device': list(range(torch.cuda.device_count()))
              if torch.cuda.device_count() > 1 else 0
}

# Start training
results = model.train(**train_args)

print("Training complete.")

# Model Export 
# Paths
folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/10_epoch_training_output"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")

### **No. of Epochs : 15 Epoch**

In [ ]:
from ultralytics import YOLO
import os
import shutil
import torch

# Update Data.yaml path
data_yaml_path = '/kaggle/input/kaggle-yaml/kaggle_new_data.yaml'

# Define the dataset directory path
dataset_base_path = "/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset"

#cache_dir 
cache_dir = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)

# Writable folder for YOLO cache
os.makedirs("/kaggle/working/cache", exist_ok=True)

# Writable folder for PyTorch CUDA kernels
os.makedirs("/kaggle/working/torch_cache", exist_ok=True)
os.environ["TORCH_KERNEL_CACHE_PATH"] = "/kaggle/working/torch_cache"

# Initialize the YOLO model
model = YOLO('/kaggle/input/20epoch/pytorch/default/1/best.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 15, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.001,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.05,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 5, # Disable mosaic for the last 10 epochs
    'cache': cache_dir,
    'workers': 3,
    'device': list(range(torch.cuda.device_count()))
              if torch.cuda.device_count() > 1 else 0
}

# Start training
results = model.train(**train_args)

print("Training complete.")

# Model Export 
# Paths
folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/15_epoch_training_output"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")


### **No. of Epochs : 20 Epoch**

In [1]:
from ultralytics import YOLO
import os
import shutil
import torch

# Update Data.yaml path
data_yaml_path = '/kaggle/input/data-yaml/kaggle_new_data (1).yaml'

# Define the dataset directory path
dataset_base_path = "/kaggle/input/indian-driving-dataset-detections-yolov11/IDDDetectionsYOLODataset"

#cache_dir 
cache_dir = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)

# Writable folder for YOLO cache
os.makedirs("/kaggle/working/cache", exist_ok=True)

# Writable folder for PyTorch CUDA kernels
os.makedirs("/kaggle/working/torch_cache", exist_ok=True)
os.environ["TORCH_KERNEL_CACHE_PATH"] = "/kaggle/working/torch_cache"

# Initialize the YOLO model
model = YOLO('/kaggle/input/20epoch/pytorch/default/1/best.pt')

# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': '/kaggle/input/20epoch/pytorch/default/1/best.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 20, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'lr0': 0.001,
    'weight_decay': 0.0005,
    'momentum': 0.937,
    'mosaic': 1.0,
    'mixup': 0.05,
    'hsv_v': 0.4,
    'hsv_s': 0.7,
    'box': 7.5,
    'cls': 0.5,
    'dfl': 1.5,
    'optimizer': 'AdamW', # Using AdamW optimizer
    'cos_lr': True, # Cosine learning rate scheduler
    'close_mosaic': 5, # Disable mosaic for the last 10 epochs
    'cache': cache_dir,
    'workers': 3,
    'device': list(range(torch.cuda.device_count()))
              if torch.cuda.device_count() > 1 else 0
}

# Start training
results = model.train(**train_args)

print("Training complete.")

# Model Export 
# Paths
folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/20_epoch_training_output"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")


Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
                                                     CUDA:1 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=/kaggle/working/cache, cfg=None, classes=None, close_mosaic=5, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/input/data-yaml/kaggle_new_data (1).yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.05, mode=train, model=/kaggle/input/20epoch/pytorch/default/1/best.pt, momentum=0.937, mosaic=1.0, 